# Task 2 - Generative AI: Domain-Specific Fine-Tuning Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/CDAZZDEV-MLE-YourName/blob/main/task2_genai/notebooks/Task2_Finetuning.ipynb)

Use case: **Financial Compliance Policy Assistant** — see `../src/problem_statement.md`
for the full definition.

**Requires a GPU runtime** for the fine-tuning section (Runtime -> Change runtime type -> T4 or L4 GPU).
Dataset generation and evaluation cells work on CPU too.

**Before running:** add `GROQ_API_KEY` (and optionally `HF_TOKEN`) as Colab secrets.

In [5]:
# Install project dependencies in Colab before running the remaining cells.
# Uncomment the next line for a fresh Colab runtime.
# !pip install -q -r ../requirements.txt

from pathlib import Path
import json
import sys

root_candidates = [Path.cwd(), Path("/content/task2_genai"), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "src").is_dir() and (path / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Upload the complete task2_genai folder to /content/task2_genai."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
LOG_DIR = PROJECT_ROOT / "logs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/chiransiriwardena/Documents/Financial-AI/task2_genai


In [14]:
import os
try:
    from google.colab import userdata
    os.environ.setdefault("GROQ_API_KEY", userdata.get("GROQ_API_KEY") or "")
    os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY") or "")
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass

os.environ["TEACHER_PROVIDER"] = "openrouter"
os.environ["TEACHER_MODEL_OPENROUTER"] = "openai/gpt-4o-mini"
os.environ["JUDGE_PROVIDER"] = "groq"

print(f"Teacher provider: {os.environ['TEACHER_PROVIDER']}")
print(f"Teacher model: {os.environ['TEACHER_MODEL_OPENROUTER']}")
print(f"Groq key configured: {bool(os.environ.get('GROQ_API_KEY'))}")
print(f"OpenRouter key configured: {bool(os.environ.get('OPENROUTER_API_KEY'))}")


Teacher provider: openrouter
Teacher model: openai/gpt-4o-mini
Groq key configured: True
OpenRouter key configured: True


## Task 2A - Dataset Generation
Generates ~120 synthetic examples across 8 compliance topics via the teacher model, reports diversity metrics, and writes the 80/10/10 JSONL split.

In [2]:
import importlib
import os
import sys
from pathlib import Path

# Make this cell runnable after a fresh Colab restart, even if the setup cell was skipped.
root_candidates = [Path.cwd(), Path("/content/task2_genai"), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "src").is_dir() and (path / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Upload the complete task2_genai folder to /content/task2_genai.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Force OpenRouter before importing/reloading project modules.
os.environ["TEACHER_PROVIDER"] = "openrouter"
os.environ["TEACHER_MODEL_OPENROUTER"] = "openai/gpt-4o-mini"

import src.config as config
importlib.reload(config)
import src.dataset_generation as dataset_generation
importlib.reload(dataset_generation)

run_dataset_pipeline = dataset_generation.run_dataset_pipeline
TEACHER_GENERATION_SYSTEM_PROMPT = dataset_generation.TEACHER_GENERATION_SYSTEM_PROMPT

print(f"Project root: {PROJECT_ROOT}")
print(f"Teacher provider: {config.TEACHER_PROVIDER}")
print(f"Teacher model: {config.TEACHER_MODEL_OPENROUTER}")
print("OpenRouter key configured:", bool(config.OPENROUTER_API_KEY))


Project root: /Users/chiransiriwardena/Documents/Financial-AI/task2_genai
Teacher provider: openrouter
Teacher model: openai/gpt-4o-mini
OpenRouter key configured: True


In [17]:
result = run_dataset_pipeline()
print(f"Total examples: {result['total']}")
print(f"Train / Val / Test sizes: {result['train_size']} / {result['val_size']} / {result['test_size']}")


INFO:src.dataset_generation:Generating 15 examples for topic=anti_money_laundering_kyc
INFO:src.dataset_generation:Generating 15 examples for topic=personal_account_dealing_insider_trading
INFO:src.dataset_generation:Generating 15 examples for topic=gifts_and_entertainment
INFO:src.dataset_generation:Generating 15 examples for topic=conflicts_of_interest
INFO:src.dataset_generation:Generating 15 examples for topic=market_abuse_information_barriers
INFO:src.dataset_generation:Generating 15 examples for topic=whistleblowing
INFO:src.dataset_generation:Generating 15 examples for topic=data_privacy_client_confidentiality
INFO:src.dataset_generation:Generating 15 examples for topic=outside_business_activities
INFO:src.dataset_generation:Dataset written: train=88 val=11 test=12 (total=111)


Total examples: 111
Train / Val / Test sizes: 88 / 11 / 12


In [18]:
import json
print(json.dumps(result["diversity_report"], indent=2))

{
  "total_examples": 111,
  "topic_distribution": {
    "anti_money_laundering_kyc": 14,
    "personal_account_dealing_insider_trading": 14,
    "gifts_and_entertainment": 14,
    "conflicts_of_interest": 14,
    "market_abuse_information_barriers": 14,
    "whistleblowing": 14,
    "data_privacy_client_confidentiality": 14,
    "outside_business_activities": 13
  },
  "num_distinct_topics": 8,
  "prompt_length_words": {
    "min": 31,
    "max": 80,
    "mean": 49.810810810810814
  },
  "top_keywords": [
    [
      "must",
      117
    ],
    [
      "any",
      86
    ],
    [
      "employees",
      68
    ],
    [
      "compliance",
      64
    ],
    [
      "report",
      50
    ],
    [
      "client",
      45
    ],
    [
      "from",
      45
    ],
    [
      "business",
      44
    ],
    [
      "within",
      43
    ],
    [
      "days",
      37
    ],
    [
      "what",
      35
    ],
    [
      "can",
      35
    ],
    [
      "department",
      35
 

**Diversity check:** confirm `num_distinct_topics == 8`, that `prompt_length_words` shows a
real spread (not all identical), and that `top_keywords` isn't dominated by a single repeated
scenario before proceeding — per the spec, a homogeneous dataset scores zero on this criterion
regardless of size.

## Task 2B - QLoRA Fine-Tuning
Every hyperparameter below is justified in `src/config.py`.

In [19]:
from src import config

hparams = {
    "base_model": config.STUDENT_BASE_MODEL,
    "lora_r": config.LORA_R,
    "lora_alpha": config.LORA_ALPHA,
    "lora_dropout": config.LORA_DROPOUT,
    "target_modules": config.LORA_TARGET_MODULES,
    "learning_rate": config.LEARNING_RATE,
    "lr_scheduler": config.LR_SCHEDULER_TYPE,
    "warmup_ratio": config.WARMUP_RATIO,
    "epochs": config.NUM_EPOCHS,
    "per_device_batch_size": config.PER_DEVICE_TRAIN_BATCH_SIZE,
    "grad_accum_steps": config.GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": config.PER_DEVICE_TRAIN_BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": config.MAX_SEQ_LENGTH,
    "quantization": f"{config.BNB_4BIT_QUANT_TYPE} 4-bit, double_quant={config.BNB_4BIT_USE_DOUBLE_QUANT}",
}
print(json.dumps(hparams, indent=2))


{
  "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "learning_rate": 0.0002,
  "lr_scheduler": "cosine",
  "warmup_ratio": 0.03,
  "epochs": 3,
  "per_device_batch_size": 2,
  "grad_accum_steps": 8,
  "effective_batch_size": 16,
  "max_seq_length": 1024,
  "quantization": "nf4 4-bit, double_quant=True"
}


In [ ]:
import importlib
import src.finetune

importlib.reload(src.finetune)
from src.finetune import run_finetuning

# use_wandb=True if you've set up a free Weights & Biases account and want the
# dashboard view in addition to the manual logs/training_loss.json below.
save_paths = run_finetuning(use_wandb=False)
print(save_paths)


INFO:datasets:TensorFlow version 2.20.0 available.
INFO:datasets:JAX version 0.6.2 available.
/Users/chiransiriwardena/miniforge3/envs/ml/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


: 

In [ ]:
with open(LOG_DIR / "training_loss.json") as f:
    loss_history = json.load(f)
for entry in loss_history:
    print(entry)

import matplotlib.pyplot as plt
epochs = [e["epoch"] for e in loss_history]
train_losses = [e["train_loss"] for e in loss_history]
val_losses = [e["val_loss"] for e in loss_history if e["val_loss"] is not None]
plt.plot(epochs, train_losses, label="train_loss")
if val_losses:
    plt.plot(epochs[:len(val_losses)], val_losses, label="val_loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Training / Validation Loss")
plt.show()


### Push the merged model to Hugging Face Hub

Per Section 3, the Task 2 deliverable is a Hugging Face model repo (public preferred;
if private, grant the reviewer account read access). This uses only the free Hub
tier - no paid infra.

In [ ]:
import importlib
import src.config
import src.push_to_hub

importlib.reload(src.config)
importlib.reload(src.push_to_hub)
from src.push_to_hub import push

# Requires HF_TOKEN and HF_HUB_MODEL_ID as environment variables or Colab secrets.
# Use private=True to restrict repository access.
push(private=False)


## Task 2C - Evaluation and Baseline Comparison

In [ ]:
# Generate predictions from BOTH the base model (system prompt only, no fine-tuning)
# and the fine-tuned merged model, served sequentially to stay within GPU memory.

import gc
import importlib
import json
import torch

import src.inference
importlib.reload(src.inference)
from src.inference import query_base_model, query_finetuned_model, free_cached_models


def clear_gpu_memory():
    free_cached_models()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


test_examples = [json.loads(line) for line in (DATA_DIR / "test.jsonl").read_text().splitlines()]
print(f"Loaded {len(test_examples)} held-out test examples")

clear_gpu_memory()
base_predictions_raw = [query_base_model(example) for example in test_examples]
clear_gpu_memory()
finetuned_predictions_raw = [query_finetuned_model(example) for example in test_examples]
clear_gpu_memory()

base_predictions = [json.dumps(prediction) for prediction in base_predictions_raw]
finetuned_predictions = [json.dumps(prediction) for prediction in finetuned_predictions_raw]
references = [example["messages"][-1]["content"] for example in test_examples]


In [ ]:
from src.evaluate import build_rouge_comparison_table, summarize_rouge_table

rouge_rows = build_rouge_comparison_table(base_predictions, finetuned_predictions, references)
summary = summarize_rouge_table(rouge_rows)
print(json.dumps(summary, indent=2))

import pandas as pd
pd.DataFrame([r.model_dump() for r in rouge_rows])


In [ ]:
from src.evaluate import judge_response

judge_scores = []
for ex, pred_raw in zip(test_examples, finetuned_predictions_raw):
    user_msg = ex["messages"][1]["content"]
    reference = json.loads(ex["messages"][2]["content"])
    score = judge_response(
        policy_excerpt=user_msg,
        employee_question=user_msg,
        reference=reference,
        model_response=pred_raw,
    )
    judge_scores.append(score)
    print(score)

valid_scores = [s for s in judge_scores if s is not None]
if valid_scores:
    avg_overall = sum(s.overall for s in valid_scores) / len(valid_scores)
    print(f"\nAverage judge 'overall' score: {avg_overall:.2f} / 5 (n={len(valid_scores)})")


In [ ]:
from src.evaluate import save_manual_review_template, compute_hallucination_rate
from src.schemas import ManualReviewEntry
import csv

save_manual_review_template(finetuned_predictions, OUTPUT_DIR / "manual_review.csv")
print("Now open outputs/manual_review.csv and label at least 10 rows by hand")
print("(correct / partially_correct / hallucinated), save it, then re-run this cell.")

reviews = []
with open(OUTPUT_DIR / "manual_review.csv") as f:
    for row in csv.DictReader(f):
        label = row["label (correct/partially_correct/hallucinated)"].strip()
        if label:
            reviews.append(ManualReviewEntry(
                example_id=int(row["example_id"]),
                model_output=row["model_output"],
                label=label,
                notes=row.get("notes") or None,
            ))

print(compute_hallucination_rate(reviews))


### Qualitative analysis (fill in after running the cells above)

**Where fine-tuning improved behaviour:** _replace with 2-3 specific before/after
examples from your own test-set run._

**Remaining failure modes and next steps:** _replace with the patterns you actually
observed in `outputs/manual_review.csv`._

## Bonus - RAG Fallback Layer

In [ ]:
from src.rag_fallback import build_policy_vector_store, rag_fallback_query, needs_rag_fallback
from src.inference import query_finetuned_model_with_context

# Build the vector store from the full policy excerpts used across the test set
policy_documents = [
    {"id": f"doc_{i}", "topic": f"topic_{i}", "text": ex["messages"][1]["content"]}
    for i, ex in enumerate(test_examples)
]
collection = build_policy_vector_store(policy_documents)

# Find a low-confidence prediction from the fine-tuned model run above and re-query with retrieval
low_conf_pairs = [(ex, pred) for ex, pred in zip(test_examples, finetuned_predictions_raw) if needs_rag_fallback(pred)]
print(f"{len(low_conf_pairs)} of {len(test_examples)} fine-tuned predictions were low-confidence.")

if low_conf_pairs:
    example, original_response = low_conf_pairs[0]
    original_excerpt = example["messages"][1]["content"]
    employee_question = original_excerpt.split("Employee question:")[-1]
    improved = rag_fallback_query(
        collection, employee_question, original_excerpt, original_response,
        query_model_fn=query_finetuned_model_with_context,
    )
    print("Before:", original_response)
    print("After:", improved)
else:
    print("No low-confidence predictions to demonstrate fallback on for this run.")


## Summary

- **Task 2A**: 120 synthetic compliance examples generated across 8 topics via a
  70B teacher model, diversity-checked, formatted as chat-template JSONL, split 80/10/10.
- **Task 2B**: Mistral-7B-Instruct fine-tuned with QLoRA (4-bit NF4), every
  hyperparameter justified in `config.py`, per-epoch loss logged, adapter merged
  and saved.
- **Task 2C**: ROUGE-L base-vs-fine-tuned comparison, LLM-as-judge structured
  scoring, and a manually reviewed hallucination rate.
- **Bonus**: ChromaDB RAG fallback triggered on low self-reported confidence.

See `../README.md` for the full requirement-to-code mapping and `../CITATIONS.md` /
`../REFLECTION.md` for AI-usage disclosure and architectural reflection.